# Feature Engineering and Derived Metrics
This notebook translates the legacy `feature andMore.ipynb` workflow into the main pipeline. It preserves the pedagogical commentary in English, computes derived energy ratios, and prepares the final feature set for model training.

## Objectives

- Review the cleaned dataset output of `01_clean_data.ipynb`.
- Explain the meaning of key energy and emissions variables.
- Calculate electricity share and validate its role in the model.
- Apply log-transformations and outlier detection for energy and surface.
- Build the final feature dataset using shared routines.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.append(str(Path('src').resolve()))
from seattle_energy.data_processing import build_feature_dataframe, save_processed_data

plt.style.use('seaborn-v0_8')


## Energy units and greenhouse gas emissions

This section explains the main target and related variables from the legacy notebook. It keeps the educational commentary while focusing on the real model goal.

### What is `SiteEnergyUse(kBtu)`?
`SiteEnergyUse(kBtu)` is the building's annual total energy consumption in thousands of British thermal units.
- 1 kBtu = 1,000 British thermal units
- 1 kBtu ≈ 0.293 kWh

This is the actual energy reading we want to predict, not a climate-adjusted estimate.

### What is `SiteEnergyUseWN(kBtu)`?
`SiteEnergyUseWN(kBtu)` is weather-normalized site energy use. It removes the influence of unusual weather so that buildings can be compared more fairly.

However, we do not use it here because:
1. It is not directly observable before the fact.
2. It already includes a statistical adjustment, which may leak information.
3. Our model aims to predict real, billed energy use, not a hypothetical normalized value.

### What is `TotalGHGEmissions`?
`TotalGHGEmissions` measures the annual greenhouse gas emissions from the building, usually in metric tons of CO₂ equivalent.

This value is derived from the building's energy use and the emission factors for each energy source. It is useful for understanding the environmental footprint, but our primary target remains energy consumption.

## 1. Load the cleaned dataset

This notebook expects the cleaned dataset saved by `01_clean_data.ipynb`.

In [ ]:
clean_path = Path('data/2016_Building_Energy_Benchmarking_ML.csv')
if not clean_path.exists():
    raise FileNotFoundError(f'Expected cleaned dataset not found: {clean_path}')
df = pd.read_csv(clean_path, low_memory=False)
print('Loaded cleaned dataset:', clean_path)
print('Rows:', len(df), 'Columns:', len(df.columns))


In [ ]:
df.head()


## 2. Electricity share and derived ratio

This section uses the legacy `feature andMore` logic to compute how much of total energy use comes from electricity.

In [ ]:
electricity_cols = [col for col in df.columns if 'Electricity' in col and 'kBtu' in col]
print('Electricity-related columns found:', electricity_cols)
total_energy_col = 'SiteEnergyUse(kBtu)'
df['Electricity_Use_Total'] = df[electricity_cols].sum(axis=1, skipna=True)
df['Electricity_Proportion'] = df['Electricity_Use_Total'] / df[total_energy_col]
df_valid_ratio = df[df['Electricity_Proportion'].between(0, 1)]
print('Valid rows for electricity proportion:', len(df_valid_ratio))
df_valid_ratio['Electricity_Proportion'].describe(percentiles=[.25, .5, .75, .9, .95])


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(y=df_valid_ratio['Electricity_Proportion'], color='skyblue')
plt.title('Electricity proportion of total energy use')
plt.ylabel('Electricity proportion (0 to 1)')
plt.grid(True)
plt.tight_layout()
plt.show()


## 3. Log transforms and outlier detection

This section applies the legacy `feature andMore` analysis for log energy, log surface, and outlier identification.

In [ ]:
analysis_df = df[[total_energy_col, 'PropertyGFABuilding(s)']].dropna()
analysis_df = analysis_df[(analysis_df[total_energy_col] > 0) & (analysis_df['PropertyGFABuilding(s)'] > 0)]
analysis_df['Log_Energy'] = np.log10(analysis_df[total_energy_col] + 1)
analysis_df['Log_Surface'] = np.log10(analysis_df['PropertyGFABuilding(s)'] + 1)
print('Rows used for log/ outlier analysis:', len(analysis_df))


In [ ]:
plt.figure(figsize=(16, 10))
plt.subplot(2, 2, 1)
sns.histplot(analysis_df['Log_Energy'], kde=True, bins=30, color='skyblue')
plt.title('Histogram + KDE of Log(Energy)')
plt.subplot(2, 2, 2)
sns.boxplot(x=analysis_df['Log_Energy'], color='lightgreen')
plt.title('Boxplot of Log(Energy)')
plt.subplot(2, 2, 3)
stats.probplot(analysis_df['Log_Energy'], dist='norm', plot=plt)
plt.title('Q-Q Plot of Log(Energy)')
plt.subplot(2, 2, 4)
sns.kdeplot(analysis_df['Log_Energy'], label='Empirical density', fill=True)
mean = analysis_df['Log_Energy'].mean()
std = analysis_df['Log_Energy'].std()
x = np.linspace(analysis_df['Log_Energy'].min(), analysis_df['Log_Energy'].max(), 100)
plt.plot(x, stats.norm.pdf(x, mean, std), label='Gaussian density', linestyle='--')
plt.title('Empirical density vs. normal distribution')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
Q1 = analysis_df['Log_Energy'].quantile(0.25)
Q3 = analysis_df['Log_Energy'].quantile(0.75)
IQR = Q3 - Q1
low = Q1 - 1.5 * IQR
high = Q3 + 1.5 * IQR
analysis_df['outlier_energy'] = (analysis_df['Log_Energy'] < low) | (analysis_df['Log_Energy'] > high)
analysis_df['outlier_surface'] = analysis_df['Log_Surface'] > analysis_df['Log_Surface'].quantile(0.95)
print('Energy outliers:', analysis_df['outlier_energy'].sum())
print('Surface top 5%:', analysis_df['outlier_surface'].sum())
analysis_df[['Log_Energy','Log_Surface','outlier_energy','outlier_surface']].head()


## 4. Build the feature dataset

Now use the shared code in `src/seattle_energy/data_processing.py` to create the final model features and save them for BentoML.

In [ ]:
processed = build_feature_dataframe(df)
print(f'Processed dataset has {len(processed)} rows and {len(processed.columns)} columns')
processed.head()


In [ ]:
save_processed_data(processed)
print('Saved processed dataset to data/processed/feature_engineered_cleaned_for_bento.csv')
